# Pneumonia Detection & Localization — Member 2
## Model Training & Evaluation Pipeline

**Member 2 Scope & Deliverables:**
- **Architecture:** DenseNet121 adapted for binary pneumonia classification (single output logit with `BCEWithLogitsLoss`).
- **Optimization:** Adam optimizer (`lr=1e-4`, `weight_decay=1e-5`).
- **Model Selection:** Track validation loss and **Validation ROC-AUC** across all 15 epochs. Save `best_model.pth` strictly whenever validation ROC-AUC improves.
- **Held-Out Test Set:** Do **NOT** use the test set for model selection. Evaluate once on `test.csv` only after model training and selection are fully finalized.
- **Deliverables:** Final training logs, `best_model.pth` checkpoint, test evaluation metrics (Accuracy, Precision, Recall, F1, ROC-AUC), confusion matrix, and ROC curve.

### Step 1: Check GPU Acceleration
Confirm GPU availability (**Runtime -> Change runtime type -> T4 GPU**).

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive
Mount Google Drive to persist checkpoints, metadata, and evaluation results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ensure persistent project directories exist in Google Drive
!mkdir -p /content/drive/MyDrive/Pneumonia_Project/checkpoints
!mkdir -p /content/drive/MyDrive/Pneumonia_Project/metadata
!mkdir -p /content/drive/MyDrive/Pneumonia_Project/results

### Step 3: Clone Repository and Switch to `feature/model`

In [ ]:
%cd /content
!rm -rf Pneumonia_Detection_Localization
!git clone https://github.com/Tanujn06/Pneumonia_Detection_Localization.git
%cd /content/Pneumonia_Detection_Localization
!git checkout feature/model
!git pull origin feature/model

# Sync genuine metadata patient splits to Google Drive
!cp -u metadata/*.csv /content/drive/MyDrive/Pneumonia_Project/metadata/
print('Metadata verified:')
!ls -lh /content/drive/MyDrive/Pneumonia_Project/metadata/

### Step 4: Install Dependencies & Extract Images
Extract RSNA DICOM images into Colab fast local SSD storage (`/content/rsna_train_images/`).

In [ ]:
!pip install -q pydicom

import os
img_dir = '/content/rsna_train_images/stage_2_train_images'
os.makedirs('/content/rsna_train_images', exist_ok=True)

# Check available paths for stage_2_train_images.zip
zip_candidates = [
    '/content/drive/MyDrive/dataset/stage_2_train_images.zip',
    '/content/drive/MyDrive/Pneumonia_Project/stage_2_train_images.zip',
    '/content/drive/MyDrive/stage_2_train_images.zip'
]

zip_found = None
for z in zip_candidates:
    if os.path.exists(z):
        zip_found = z
        break

if not os.path.exists(img_dir) or len(os.listdir(img_dir)) == 0:
    if zip_found:
        print(f'Extracting images from: {zip_found} ...')
        !unzip -q '{zip_found}' -d /content/rsna_train_images/
        print('Extraction complete!')
    else:
        print('Notice: No stage_2_train_images.zip found in default Drive paths.')
        print('Please ensure your dataset folder has a shortcut in My Drive, or upload the ZIP.')
else:
    print(f'Images already extracted: {len(os.listdir(img_dir))} files.')

### Step 5: Run Full Training Pipeline (15 Epochs)
- Trains DenseNet121 with `BCEWithLogitsLoss` and `Adam`.
- Tracks **Train Loss**, **Train Acc**, **Val Loss**, **Val Acc**, and **Validation ROC-AUC** for every epoch.
- Automatically saves `best_model.pth` whenever Validation ROC-AUC strictly improves.
- Checkpoint and full logs are persisted directly to Google Drive.

In [ ]:
%cd /content/Pneumonia_Detection_Localization
!python rc/src/train.py \
    --train_csv '/content/drive/MyDrive/Pneumonia_Project/metadata/train.csv' \
    --val_csv '/content/drive/MyDrive/Pneumonia_Project/metadata/val.csv' \
    --img_dir '/content/rsna_train_images/stage_2_train_images/' \
    --checkpoint_dir '/content/drive/MyDrive/Pneumonia_Project/checkpoints' \
    --epochs 15 \
    --batch_size 32 \
    --lr 0.0001 \
    --num_workers 2

### Step 6: Verify Saved Checkpoint & Final Training Logs
Confirm the saved epoch, validation ROC-AUC, and parameters of `best_model.pth`, then display the full training history.

In [ ]:
# ==============================================================================
# ALL-IN-ONE: VERIFY CHECKPOINT & RUN FINAL HELD-OUT TEST EVALUATION
# ==============================================================================
import os, json
from google.colab import drive
from IPython.display import Image, display

# 1. Mount Google Drive
print(">>> [1/5] Mounting Google Drive...")
drive.mount("/content/drive")

# 2. Ensure repository is available
print("
>>> [2/5] Setting up project repository...")
%cd /content
if not os.path.exists("/content/Pneumonia_Detection_Localization"):
    !git clone https://github.com/Tanujn06/Pneumonia_Detection_Localization.git
%cd /content/Pneumonia_Detection_Localization
!git checkout feature/model
!git pull origin feature/model

# 3. Check and extract images if needed
img_dir = "/content/rsna_train_images/stage_2_train_images"
if not os.path.exists(img_dir) or len(os.listdir(img_dir)) == 0:
    print("
>>> Extracting images for evaluation...")
    zip_path = None
    for candidate in [
        "/content/drive/MyDrive/Pneumonia_Project/stage_2_train_images.zip",
        "/content/drive/MyDrive/dataset/stage_2_train_images.zip",
        "/content/drive/MyDrive/stage_2_train_images.zip"
    ]:
        if os.path.exists(candidate):
            zip_path = candidate
            break
    if zip_path:
        !mkdir -p /content/rsna_train_images
        !unzip -q "{zip_path}" -d /content/rsna_train_images/
        print("Images ready!")

# 4. Verify saved best_model.pth checkpoint
print("
>>> [3/5] Verifying best_model.pth Checkpoint...")
!python rc/src/verify_checkpoint.py \n    --checkpoint "/content/drive/MyDrive/Pneumonia_Project/checkpoints/best_model.pth"

# Display training logs
log_file = "/content/drive/MyDrive/Pneumonia_Project/checkpoints/training_logs.txt"
if os.path.exists(log_file):
    print("
--- Training Logs Across Epochs ---")
    with open(log_file, "r") as f:
        print(f.read())

# 5. Run Held-Out Test Evaluation (Generates fresh metrics!)
print("
>>> [4/5] Evaluating on held-out test.csv...")
!python rc/src/evaluate.py \n    --test_csv "/content/drive/MyDrive/Pneumonia_Project/metadata/test.csv" \n    --img_dir "/content/rsna_train_images/stage_2_train_images/" \n    --checkpoint "/content/drive/MyDrive/Pneumonia_Project/checkpoints/best_model.pth" \n    --results_dir "/content/drive/MyDrive/Pneumonia_Project/results" \n    --batch_size 32

# 6. Display Final Updated Metrics & Plots
print("
>>> [5/5] Final Updated Test Results:")
results_dir = "/content/drive/MyDrive/Pneumonia_Project/results"
metrics_file = f"{results_dir}/test_metrics.json"

if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        metrics = json.load(f)

    print("="*50)
    print("       FINAL HELD-OUT TEST EVALUATION RESULTS")
    print("="*50)
    for k, v in metrics.items():
        if k != "confusion_matrix":
            print(f"{k:20s}: {v:.4f}")
    print("="*50)

    print("
--- Confusion Matrix ---")
    if os.path.exists(f"{results_dir}/confusion_matrix.png"):
        display(Image(filename=f"{results_dir}/confusion_matrix.png"))

    print("
--- ROC Curve ---")
    if os.path.exists(f"{results_dir}/roc_curve.png"):
        display(Image(filename=f"{results_dir}/roc_curve.png"))


### Step 7: Evaluate Once on Held-Out `test.csv`
> **Important:** As specified in the rubric, model selection was strictly performed on the validation set. We now evaluate the chosen `best_model.pth` once on the held-out test set to assess generalization.

In [ ]:
%cd /content/Pneumonia_Detection_Localization
!python rc/src/evaluate.py \
    --test_csv '/content/drive/MyDrive/Pneumonia_Project/metadata/test.csv' \
    --img_dir '/content/rsna_train_images/stage_2_train_images/' \
    --checkpoint '/content/drive/MyDrive/Pneumonia_Project/checkpoints/best_model.pth' \
    --results_dir '/content/drive/MyDrive/Pneumonia_Project/results' \
    --batch_size 32

### Step 8: Display Final Test Metrics, Confusion Matrix, and ROC Curve

In [ ]:
import json
from IPython.display import Image, display

results_dir = '/content/drive/MyDrive/Pneumonia_Project/results'
metrics_file = f'{results_dir}/test_metrics.json'

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)

    print('='*50)
    print('       FINAL HELD-OUT TEST EVALUATION RESULTS')
    print('='*50)
    for k, v in metrics.items():
        if k != 'confusion_matrix':
            print(f'{k:20s}: {v:.4f}')
    print('='*50)

    print('\n--- Confusion Matrix ---')
    display(Image(filename=f'{results_dir}/confusion_matrix.png'))

    print('\n--- ROC Curve ---')
    display(Image(filename=f'{results_dir}/roc_curve.png'))
else:
    print('Results not found. Please run Step 7 first.')